In [2]:
import os
from glob import glob
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from IPython.display import display
import cv2
from tqdm import tqdm
import shutil
import xml.etree.ElementTree as ET

## Collect the data and split train set vs test set

In [4]:
INPUT_PATH = os.path.join(os.getcwd(), 'data')
OUTPUT_PATH = os.path.join(os.getcwd(), 'data_processed')
os.makedirs(OUTPUT_PATH, exist_ok=True)
TRAIN_ANNOTATIONS_DIR = os.path.join(INPUT_PATH, 'DETRAC-Train-Annotations-XML', 'DETRAC-Train-Annotations-XML')
TEST_ANNOTATIONS_DIR = os.path.join(INPUT_PATH, 'DETRAC-Test-Annotations-XML', 'DETRAC-Test-Annotations-XML')
ALL_IMAGES_DIR = os.path.join(INPUT_PATH, 'DETRAC-Images', 'DETRAC-Images')

In [5]:
# Split the dataset into training and testing sets based on annotations
train_annotation_files = glob(os.path.join(TRAIN_ANNOTATIONS_DIR, '*.xml'))
test_annotation_files = glob(os.path.join(TEST_ANNOTATIONS_DIR, '*.xml'))
train_image_files = []
test_image_files = []
for ann_file in train_annotation_files:
    # Use filename as sequence name instead of parsing XML
    seq_name = os.path.splitext(os.path.basename(ann_file))[0]
    img_dir = os.path.join(ALL_IMAGES_DIR, seq_name)
    if not os.path.isdir(img_dir):
        print(f"Warning: Image directory not found for sequence '{seq_name}' at {img_dir}")
        continue
    img_files = sorted(glob(os.path.join(img_dir, '*.jpg')))
    print(f"Found {len(img_files)} images in {seq_name}")
    train_image_files.extend(img_files)
for ann_file in test_annotation_files:
    # Use filename as sequence name instead of parsing XML
    seq_name = os.path.splitext(os.path.basename(ann_file))[0]
    img_dir = os.path.join(ALL_IMAGES_DIR, seq_name)
    if not os.path.isdir(img_dir):
        print(f"Warning: Image directory not found for sequence '{seq_name}' at {img_dir}")
        continue
    img_files = sorted(glob(os.path.join(img_dir, '*.jpg')))
    print(f"Found {len(img_files)} images in {seq_name}")
    test_image_files.extend(img_files)

# Copy images to processed directory preserving sequence structure
train_output_dir = os.path.join(OUTPUT_PATH, 'train_images')
test_output_dir = os.path.join(OUTPUT_PATH, 'test_images')
os.makedirs(train_output_dir, exist_ok=True)
os.makedirs(test_output_dir, exist_ok=True)

# Copy training images with directory structure
for ann_file in tqdm(train_annotation_files, desc='Copying training sequences'):
    seq_name = os.path.splitext(os.path.basename(ann_file))[0]
    source_dir = os.path.join(ALL_IMAGES_DIR, seq_name)
    dest_dir = os.path.join(train_output_dir, seq_name)
    
    if os.path.isdir(source_dir):
        os.makedirs(dest_dir, exist_ok=True)
        img_files = sorted(glob(os.path.join(source_dir, '*.jpg')))
        for img_file in img_files:
            shutil.copy(img_file, os.path.join(dest_dir, os.path.basename(img_file)))

# Copy testing images with directory structure
for ann_file in tqdm(test_annotation_files, desc='Copying testing sequences'):
    seq_name = os.path.splitext(os.path.basename(ann_file))[0]
    source_dir = os.path.join(ALL_IMAGES_DIR, seq_name)
    dest_dir = os.path.join(test_output_dir, seq_name)
    
    if os.path.isdir(source_dir):
        os.makedirs(dest_dir, exist_ok=True)
        img_files = sorted(glob(os.path.join(source_dir, '*.jpg')))
        for img_file in img_files:
            shutil.copy(img_file, os.path.join(dest_dir, os.path.basename(img_file)))

# Summary
print(f'Total training images: {len(train_image_files)}')
print(f'Total testing images: {len(test_image_files)}')

Found 664 images in MVI_20011
Found 936 images in MVI_20012
Found 437 images in MVI_20032
Found 784 images in MVI_20033
Found 800 images in MVI_20034
Found 800 images in MVI_20035
Found 906 images in MVI_20051
Found 694 images in MVI_20052
Found 800 images in MVI_20061
Found 800 images in MVI_20062
Found 800 images in MVI_20063
Found 800 images in MVI_20064
Found 1200 images in MVI_20065
Found 1660 images in MVI_39761
Found 570 images in MVI_39771
Found 1865 images in MVI_39781
Found 885 images in MVI_39801
Found 1070 images in MVI_39811
Found 880 images in MVI_39821
Found 1420 images in MVI_39851
Found 745 images in MVI_39861
Found 1270 images in MVI_39931
Found 1645 images in MVI_40131
Found 1600 images in MVI_40141
Found 1750 images in MVI_40152
Found 1490 images in MVI_40161
Found 1765 images in MVI_40162
Found 1150 images in MVI_40171
Found 2635 images in MVI_40172
Found 1700 images in MVI_40181
Found 2495 images in MVI_40191
Found 2195 images in MVI_40192
Found 925 images in MVI_

Copying testing sequences: 100%|██████████| 40/40 [02:43<00:00,  4.09s/it]

Total training images: 83791
Total testing images: 56340


## Pre-process the data: create the images and labels folders for both training and testing set

In [6]:
CLASS_MAPPING = {
    'car': 0,
    'bus': 1,
    'van': 2,
}

def process_detrac_annotations(annotations_dir, images_path, labels_path, dataset_type):
    """
    Process DETRAC annotations and create YOLO format labels
    
    Args:
        annotations_dir: Directory containing XML annotation files
        images_path: Output directory for images (with sequence folders)
        labels_path: Output directory for labels (with sequence folders)
        dataset_type: 'train' or 'test'
    """
    os.makedirs(images_path, exist_ok=True)
    os.makedirs(labels_path, exist_ok=True)

    processed_count = 0
    
    for xml_file in tqdm(os.listdir(annotations_dir), desc=f"Processing {dataset_type} sequences"):
        if not xml_file.endswith('.xml'):
            continue
        
        sequence_name = os.path.splitext(xml_file)[0]
        image_dir = os.path.join(ALL_IMAGES_DIR, sequence_name) 
        
        if not os.path.isdir(image_dir):
            continue

        tree = ET.parse(os.path.join(annotations_dir, xml_file))
        root = tree.getroot()

        img_width, img_height = None, None
        try:
            first_image_path = os.path.join(image_dir, sorted(os.listdir(image_dir))[0])
            with Image.open(first_image_path) as img:
                img_width, img_height = img.size
        except Exception as e:
            continue
        
        # Create sequence-specific directories for images and labels
        seq_images_path = os.path.join(images_path, sequence_name)
        seq_labels_path = os.path.join(labels_path, sequence_name)
        os.makedirs(seq_images_path, exist_ok=True)
        os.makedirs(seq_labels_path, exist_ok=True)
            
        frames = root.findall('frame')
        for frame in frames:
            frame_num = int(frame.get('num'))
            image_filename = f"img{frame_num:05d}.jpg"
            label_filename = f"img{frame_num:05d}.txt"
            
            source_image_path = os.path.join(image_dir, image_filename)
            dest_image_path = os.path.join(seq_images_path, image_filename)
            dest_label_path = os.path.join(seq_labels_path, label_filename)

            if not os.path.exists(source_image_path):
                continue

            shutil.copy(source_image_path, dest_image_path)
            
            yolo_annotations = []
            target_list = frame.find('target_list')
            if target_list is not None:
                for target in target_list.findall('target'):
                    box = target.find('box')
                    attribute = target.find('attribute')
                    
                    vehicle_type = attribute.get('vehicle_type')
                    if vehicle_type not in CLASS_MAPPING:
                        continue
                    
                    class_id = CLASS_MAPPING[vehicle_type]
                    xmin = float(box.get('left'))
                    ymin = float(box.get('top'))
                    width = float(box.get('width'))
                    height = float(box.get('height'))
                    
                    x_center = (xmin + width / 2) / img_width
                    y_center = (ymin + height / 2) / img_height
                    w_norm = width / img_width
                    h_norm = height / img_height
                    
                    yolo_annotations.append(f"{class_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")

            with open(dest_label_path, 'w') as f:
                f.write('\n'.join(yolo_annotations))
            processed_count += 1

    print(f"\n{dataset_type.capitalize()} - Total images and labels successfully processed: {processed_count}")
    return processed_count

# Process training set
train_images_path = os.path.join(OUTPUT_PATH, 'train', 'images')
train_labels_path = os.path.join(OUTPUT_PATH, 'train', 'labels')
train_count = process_detrac_annotations(
    TRAIN_ANNOTATIONS_DIR, 
    train_images_path, 
    train_labels_path, 
    'train'
)

# Process testing set
test_images_path = os.path.join(OUTPUT_PATH, 'test', 'images')
test_labels_path = os.path.join(OUTPUT_PATH, 'test', 'labels')
test_count = process_detrac_annotations(
    TEST_ANNOTATIONS_DIR, 
    test_images_path, 
    test_labels_path, 
    'test'
)

print(f"\n✅ Conversion complete!")
print(f"Training: {train_count} images with labels")
print(f"Testing: {test_count} images with labels")


Processing train sequences: 100%|██████████| 60/60 [05:13<00:00,  5.23s/it]



Train - Total images and labels successfully processed: 82085


Processing test sequences: 100%|██████████| 40/40 [03:36<00:00,  5.42s/it]


Test - Total images and labels successfully processed: 56167

✅ Conversion complete!
Training: 82085 images with labels
Testing: 56167 images with labels


## Assign labels to images

In [7]:
# Reverse mapping for class names
CLASS_NAMES = {v: k for k, v in CLASS_MAPPING.items()}

In [8]:
def annotate_images_with_boxes(images_base_dir, labels_base_dir, output_base_dir, class_names):
    """
    Add ground-truth bounding boxes with class labels to images.
    Processes all sequences in the directory structure.
    
    Args:
        images_base_dir: Base directory containing sequence folders with images
        labels_base_dir: Base directory containing sequence folders with labels
        output_base_dir: Base directory to save annotated images
        class_names: Dictionary mapping class_id to class name
    """
    os.makedirs(output_base_dir, exist_ok=True)
    
    # Get all sequence directories
    sequences = [d for d in os.listdir(images_base_dir) if os.path.isdir(os.path.join(images_base_dir, d))]
    
    total_annotated = 0
    for seq_name in tqdm(sequences, desc="Processing sequences"):
        img_dir = os.path.join(images_base_dir, seq_name)
        labels_dir = os.path.join(labels_base_dir, seq_name)
        output_dir = os.path.join(output_base_dir, seq_name)
        os.makedirs(output_dir, exist_ok=True)
        
        image_files = sorted([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
        
        for img_file in image_files:
            label_file = img_file.rsplit('.', 1)[0] + '.txt'
            
            img_path = os.path.join(img_dir, img_file)
            label_path = os.path.join(labels_dir, label_file)
            output_path = os.path.join(output_dir, img_file)
            
            # Read image using method that handles Unicode paths
            img = cv2.imdecode(np.fromfile(img_path, dtype=np.uint8), cv2.IMREAD_COLOR)
            if img is None:
                continue
            
            height, width = img.shape[:2]
            
            # Read labels if file exists
            if os.path.exists(label_path):
                with open(label_path, 'r') as f:
                    lines = f.readlines()
                
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    
                    class_id = int(parts[0])
                    x_center = float(parts[1])
                    y_center = float(parts[2])
                    w_norm = float(parts[3])
                    h_norm = float(parts[4])
                    
                    # Denormalize coordinates
                    x_center_px = int(x_center * width)
                    y_center_px = int(y_center * height)
                    w_px = int(w_norm * width)
                    h_px = int(h_norm * height)
                    
                    # Calculate top-left and bottom-right corners
                    x1 = max(0, x_center_px - w_px // 2)
                    y1 = max(0, y_center_px - h_px // 2)
                    x2 = min(width - 1, x_center_px + w_px // 2)
                    y2 = min(height - 1, y_center_px + h_px // 2)
                    
                    # Draw rectangle
                    color = (0, 255, 0)  # Green color for boxes
                    thickness = 2
                    cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)
                    
                    # Add class label
                    class_name = class_names.get(class_id, f"Class {class_id}")
                    font = cv2.FONT_HERSHEY_SIMPLEX
                    font_scale = 0.6
                    font_thickness = 1
                    text_size = cv2.getTextSize(class_name, font, font_scale, font_thickness)[0]
                    
                    # Draw label background
                    text_x = x1
                    text_y = max(20, y1 - 5)
                    cv2.rectangle(img, (text_x, text_y - text_size[1] - 5), 
                                 (text_x + text_size[0] + 5, text_y + 5), color, -1)
                    
                    # Put text
                    cv2.putText(img, class_name, (text_x + 2, text_y - 2), 
                               font, font_scale, (0, 0, 0), font_thickness)
            
            # Save annotated image using method that handles Unicode paths
            is_success, buffer = cv2.imencode('.jpg', img)
            if is_success:
                buffer.tofile(output_path)
                total_annotated += 1
    
    print(f"✅ Annotated {total_annotated} images and saved to {output_base_dir}")
    return total_annotated

# Process training set
train_img_dir = os.path.join(OUTPUT_PATH, 'train', 'images')
train_labels_dir = os.path.join(OUTPUT_PATH, 'train', 'labels')
train_annotated_dir = os.path.join(OUTPUT_PATH, 'train', 'images_annotated')
train_count = annotate_images_with_boxes(train_img_dir, train_labels_dir, train_annotated_dir, CLASS_NAMES)

# Process testing set
test_img_dir = os.path.join(OUTPUT_PATH, 'test', 'images')
test_labels_dir = os.path.join(OUTPUT_PATH, 'test', 'labels')
test_annotated_dir = os.path.join(OUTPUT_PATH, 'test', 'images_annotated')
test_count = annotate_images_with_boxes(test_img_dir, test_labels_dir, test_annotated_dir, CLASS_NAMES)

print(f"\nTotal annotated - Train: {train_count}, Test: {test_count}")

Processing sequences:   0%|          | 0/60 [00:00<?, ?it/s]

Processing sequences: 100%|██████████| 60/60 [17:32<00:00, 17.55s/it]


✅ Annotated 82085 images and saved to c:\Users\antoi\OneDrive\Documents\Documents\Devoirs\Études sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project\data_processed\train\images_annotated


Processing sequences: 100%|██████████| 40/40 [14:08<00:00, 21.22s/it]

✅ Annotated 56167 images and saved to c:\Users\antoi\OneDrive\Documents\Documents\Devoirs\Études sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project\data_processed\test\images_annotated

Total annotated - Train: 82085, Test: 56167


## Create videos from the frames

In [ ]:
def create_video_from_frames(sequence_folder, dataset_type='train', fps=30, use_annotated=False):
    """
    Create a video from all frames in a sequence folder.
    
    Args:
        sequence_folder: Name of the sequence folder (e.g., 'MVI_20011')
        dataset_type: 'train' or 'test'
        fps: Frames per second for the output video
        use_annotated: If True, use annotated images; if False, use original images
    """
    # Determine the base directory for frames
    if use_annotated:
        frames_base_dir = os.path.join(OUTPUT_PATH, dataset_type, 'images_annotated')
    else:
        frames_base_dir = os.path.join(OUTPUT_PATH, dataset_type, 'images')
    
    # Get the full path to the sequence folder
    frames_dir = os.path.join(frames_base_dir, sequence_folder)
    
    if not os.path.exists(frames_dir):
        print(f"Error: Folder {frames_dir} does not exist")
        return
    
    # Create output directory for videos
    videos_dir = os.path.join(OUTPUT_PATH, dataset_type, 'annotated_videos')
    os.makedirs(videos_dir, exist_ok=True)
    
    # Define output video path
    video_name = f"{sequence_folder}_{'annotated' if use_annotated else 'original'}.mp4"
    output_video_path = os.path.join(videos_dir, video_name)
    
    # Get all image files sorted by name
    image_files = sorted([f for f in os.listdir(frames_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
    
    if not image_files:
        print(f"No images found in {frames_dir}")
        return
    
    # Read the first image to get dimensions
    first_image_path = os.path.join(frames_dir, image_files[0])
    first_frame = cv2.imdecode(np.fromfile(first_image_path, dtype=np.uint8), cv2.IMREAD_COLOR)
    
    if first_frame is None:
        print(f"Error reading first frame: {first_image_path}")
        return
    
    height, width = first_frame.shape[:2]
    
    # Create video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
    
    print(f"Creating video from {len(image_files)} frames...")
    
    for img_file in tqdm(image_files, desc=f"Processing {sequence_folder}"):
        img_path = os.path.join(frames_dir, img_file)
        frame = cv2.imdecode(np.fromfile(img_path, dtype=np.uint8), cv2.IMREAD_COLOR)
        
        if frame is not None:
            out.write(frame)
    
    out.release()
    print(f"✅ Video saved to {output_video_path}")
    return output_video_path

def create_videos_for_all_sequences(dataset_type='train', fps=25, use_annotated=True):
    """
    Create videos for all sequences in a dataset.
    
    Args:
        dataset_type: 'train' or 'test'
        fps: Frames per second for the output videos
        use_annotated: If True, use annotated images; if False, use original images
    """
    # Determine the base directory for frames
    if use_annotated:
        frames_base_dir = os.path.join(OUTPUT_PATH, dataset_type, 'images_annotated')
    else:
        frames_base_dir = os.path.join(OUTPUT_PATH, dataset_type, 'images')
    
    if not os.path.exists(frames_base_dir):
        print(f"Error: Directory {frames_base_dir} does not exist")
        return
    
    # Get all sequence folders
    sequences = sorted([d for d in os.listdir(frames_base_dir) 
                       if os.path.isdir(os.path.join(frames_base_dir, d))])
    
    if not sequences:
        print(f"No sequences found in {frames_base_dir}")
        return
    
    print(f"Found {len(sequences)} sequences to process")
    created_videos = []
    
    for seq in sequences:
        video_path = create_video_from_frames(seq, dataset_type, fps, use_annotated)
        if video_path:
            created_videos.append(video_path)
    
    print(f"\n{'='*60}")
    print(f"✅ Created {len(created_videos)} videos from {dataset_type} set")
    print(f"{'='*60}")
    return created_videos

# Create videos for all training sequences (annotated)
create_videos_for_all_sequences(dataset_type='train', fps=25, use_annotated=True)

Found 60 sequences to process
Creating video from 664 frames...


Processing MVI_20011: 100%|██████████| 664/664 [00:07<00:00, 94.60it/s] 


✅ Video saved to c:\Users\antoi\OneDrive\Documents\Documents\Devoirs\Études sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project\data_processed\train\annotated_videos\MVI_20011_annotated.mp4
Creating video from 936 frames...


Processing MVI_20012: 100%|██████████| 936/936 [00:11<00:00, 84.60it/s] 


✅ Video saved to c:\Users\antoi\OneDrive\Documents\Documents\Devoirs\Études sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project\data_processed\train\annotated_videos\MVI_20012_annotated.mp4
Creating video from 437 frames...


...

Processing MVI_40192: 100%|██████████| 2195/2195 [00:22<00:00, 95.95it/s] 


✅ Video saved to c:\Users\antoi\OneDrive\Documents\Documents\Devoirs\Études sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project\data_processed\train\annotated_videos\MVI_40192_annotated.mp4
Creating video from 925 frames...


Processing MVI_40201: 100%|██████████| 925/925 [00:09<00:00, 97.58it/s] 


✅ Video saved to c:\Users\antoi\OneDrive\Documents\Documents\Devoirs\Études sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project\data_processed\train\annotated_videos\MVI_40201_annotated.mp4
Creating video from 1225 frames...


Processing MVI_40204:  88%|████████▊ | 1082/1225 [00:11<00:01, 100.06it/s]